# Data Acquisition

In this lab, we will explore the process of data acquisition. 

In the case of *passive* network traffic analysis, there are generally two primary ways of acquiring data:
* Packet capture
* Network traffic flows (sometimes called IPFIX)

The advent of more programmability in networks is quickly changing this landscape. 

In particular, systems like Retina are now making it possible to ask more complex questions of network traffic from passive traffic capture and analysis but the general underlying traffic patterns are still based on raw packet capture.

## Background

Because packet captures are so large, it can sometimes be convenient to work with summary statistics about network traffic. Instead of the raw packets, data could represent the total number of bytes, packets, and so forth for flows. 

Raw traffic capture is thus sometimes represented as summaries of flow statistics, rather than raw packet traces. In this activity, we will *generate* the summary statistics and then think about what types of information is (and is not) available in a packet trace summary vs. a raw packet capture.

## Step 1: Load a Packet Trace

Load the packet capture from the last assignment.

In [21]:
import pandas as pd

ndf = pd.read_csv("data/netflix.csv.gz")
ndf.head(20)

,No.,Time,Source,Destination,Protocol,Length,Info
0,1,2018-02-11 08:10:00.534682,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0xed0c A fonts.gstatic.com
1,2,2018-02-11 08:10:00.534832,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0x301a AAAA fonts.gstatic.com
2,3,2018-02-11 08:10:00.539408,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x11d3 A googleads.g.doubleclic...
3,4,2018-02-11 08:10:00.541204,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x1284 AAAA googleads.g.doublec...
4,5,2018-02-11 08:10:00.545785,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,78,Standard query 0x3432 AAAA ytimg.l.google.com
5,6,2018-02-11 08:10:00.547036,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,96,Standard query 0xb756 A r4---sn-gxo5uxg-jqbe.g...
6,7,2018-02-11 08:10:00.547156,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x62ab A ssl.gstatic.com
7,8,2018-02-11 08:10:00.547249,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,74,Standard query 0x42fb A www.google.com
8,9,2018-02-11 08:10:00.853950,ns-vip-pro.paris.inria.fr,192.168.43.72,DNS,386,Standard query response 0x11d3 A 216.58.213.162
9,10,2018-02-11 08:10:00.853970,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x8756 A www.gstatic.com


## Step 2: Generate Statistics for Each Flow

A **flow** is defined as groups of packets that share the following attributes:
* Source IP Address
* Destination IP Address
* Source Port
* Destination Port
* Time interval

For now, we will ignore the time interval and group packets on the 4-tuple of source IP, source port, destination IP, and destination port.

The csv file does not have port numbers (and for many packets, such as TLS/SSL, the `Info` column does not show them either), so we read the ports from the original pcap. The pcap and the csv contain the same packets in the same order, so we can attach the ports to the csv rows by position. Packets with no TCP/UDP header (ARP, ICMP, etc.) get port `-1`.

For each flow in the packet trace, generate the following statistics for each flow:

* Number of bytes
* Number of packets
* Duration (time)

In [22]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from scapy.utils import PcapReader
from scapy.layers.inet import TCP, UDP

# Read source/destination ports from the pcap (same packets, same order as the csv)
src_ports, dst_ports = [], []
for pkt in PcapReader("data/netflix.pcap"):
    l4 = pkt.getlayer(TCP) or pkt.getlayer(UDP)
    src_ports.append(l4.sport if l4 else -1)
    dst_ports.append(l4.dport if l4 else -1)

assert len(src_ports) == len(ndf)
ndf["SrcPort"] = src_ports
ndf["DstPort"] = dst_ports

# Timestamps as seconds since the epoch
ndf["Timestamp"] = (
    np.array(ndf["Time"].str.replace(" ", "T").tolist(), dtype="datetime64[us]")
    .astype("int64") / 1e6
)

flow_keys = ["Source", "SrcPort", "Destination", "DstPort"]
grouped = ndf.groupby(flow_keys)

grouped.head()

,No.,Time,Source,Destination,Protocol,Length,Info,SrcPort,DstPort,Timestamp
0,1,2018-02-11 08:10:00.534682,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0xed0c A fonts.gstatic.com,55697,53,1.518337e+09
1,2,2018-02-11 08:10:00.534832,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0x301a AAAA fonts.gstatic.com,59884,53,1.518337e+09
2,3,2018-02-11 08:10:00.539408,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x11d3 A googleads.g.doubleclic...,61223,53,1.518337e+09
3,4,2018-02-11 08:10:00.541204,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x1284 AAAA googleads.g.doublec...,58785,53,1.518337e+09
4,5,2018-02-11 08:10:00.545785,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,78,Standard query 0x3432 AAAA ytimg.l.google.com,51938,53,1.518337e+09
...,...,...,...,...,...,...,...,...,...,...
141418,141419,2018-02-11 08:18:14.008579,104.31.113.215,192.168.43.72,HTTP,74,Continuation,80,58530,1.518337e+09
141419,141420,2018-02-11 08:18:14.008581,104.31.113.215,192.168.43.72,TCP,74,"[TCP Retransmission] 80 > 58530 [PSH, ACK] Seq...",80,58530,1.518337e+09
141421,141422,2018-02-11 08:18:14.008636,192.168.43.72,104.31.113.215,TCP,54,58530 > 80 [ACK] Seq=359 Ack=567 Win=261568 Len=0,58530,80,1.518337e+09
141437,141438,2018-02-11 08:18:16.174476,192.168.43.72,par10s29-in-f238.1e100.net,TCP,66,"58525 > 80 [FIN, ACK] Seq=850 Ack=518 Win=1310...",58525,80,1.518337e+09


### Total Number of Flows

Count the total number of flows in this trace.

In [23]:
num_flows = grouped.ngroups
print(f"Total number of flows: {num_flows}")

Total number of flows: 298


### Number of Bytes

Count the total number of bytes for each flow in the trace. 

Then, sort the flows by size, in bytes.  

What do you notice about the large flows? What do they look like?

In [24]:
flows = grouped.agg(Bytes=("Length", "sum")).reset_index()
flows.sort_values("Bytes", ascending=False).head(10)

,Source,SrcPort,Destination,DstPort,Bytes
226,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58506,52719449
228,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58512,31358435
225,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58498,25013186
227,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58507,10650725
214,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58497,2320672
215,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58499,1738169
116,192.168.43.72,58506,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,1555682
218,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58509,1389400
178,a23-57-80-120.deploy.static.akamaitechnologies...,443,192.168.43.72,58503,1290466
230,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58515,859154


### Number of Packets

Count the number of packets in each flow. 

What do you notice about these flows? Are they similar to the largest flows by bytes? Which differ?

In [25]:
flows["Packets"] = grouped.size().values
flows.sort_values("Packets", ascending=False).head(10)

,Source,SrcPort,Destination,DstPort,Bytes,Packets
226,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58506,52719449,34899
116,192.168.43.72,58506,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,1555682,22359
228,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58512,31358435,20855
225,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58498,25013186,16618
122,192.168.43.72,58512,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,805678,11487
108,192.168.43.72,58498,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,659389,9312
227,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58507,10650725,7106
117,192.168.43.72,58507,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,312079,4364
214,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58497,2320672,1583
215,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58499,1738169,1169


### Duration

Compute the duration of each flow, by taking the time of the last packet and subtracting the time of the first, for each flow.  

What are the longest flows in the trace?

In [26]:
flows["Duration"] = (grouped["Timestamp"].max() - grouped["Timestamp"].min()).values
flows.sort_values("Duration", ascending=False).head(10)

,Source,SrcPort,Destination,DstPort,Bytes,Packets,Duration
146,192.168.43.97,5353,224.0.0.251,5353,1004,9,486.705466
15,192.168.43.72,5353,224.0.0.251,5353,2344,12,486.400795
210,fe80::e6ce:8fff:fe01:4c54,5353,ff02::fb,5353,2584,12,486.400733
105,192.168.43.72,58495,ec2-34-252-77-54.eu-west-1.compute.amazonaws.com,443,3753,25,480.552744
179,ec2-34-252-77-54.eu-west-1.compute.amazonaws.com,443,192.168.43.72,58495,4884,21,480.407859
161,Apple_01:4c:54,-1,CeLinkLi_0d:2b:a7,-1,546,13,464.720888
163,CeLinkLi_0d:2b:a7,-1,Apple_01:4c:54,-1,780,13,464.720873
54,192.168.43.72,58232,217.146.8.3,5938,2568,32,461.828869
160,217.146.8.3,5938,192.168.43.72,58232,2530,24,459.632482
195,fd3c:8eef:8a02:0:7110:c44b:cd2b:15ff,5353,ff02::fb,5353,903,8,453.288303


## Bytes and Packets Per Second

Compute the bytes per second and packets per second for each flow.

For a simple feature computation, compute the average bytes and packets per second for each flow, for the entire duration of the flow.  If you want to get more clever or fancy, you can do "windowed averages", computing bytes or packets per second for shorter time intervals.

In [27]:
# Flows with a single packet (or identical timestamps) have zero duration -> NaN rate
dur = flows["Duration"].replace(0, np.nan)
flows["BytesPerSec"] = flows["Bytes"] / dur
flows["PacketsPerSec"] = flows["Packets"] / dur
flows.sort_values("BytesPerSec", ascending=False).head(10)

,Source,SrcPort,Destination,DstPort,Bytes,Packets,Duration,BytesPerSec,PacketsPerSec
5,192.168.1.159,-1,192.168.43.72,-1,752,8,0.000827,909229.347939,9672.652638
227,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58507,10650725,7106,49.454091,215365.903389,143.688820
226,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58506,52719449,34899,432.092108,122009.747511,80.767502
225,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58498,25013186,16618,210.397020,118885.647658,78.984008
228,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58512,31358435,20855,277.847615,112861.990914,75.059129
224,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58523,308947,216,6.268617,49284.715305,34.457362
218,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58509,1389400,929,63.603947,21844.556308,14.606012
214,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58497,2320672,1583,114.381312,20288.908753,13.839673
296,par10s38-in-f3.1e100.net,443,192.168.43.72,58526,77751,70,4.152841,18722.363388,16.855930
215,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58499,1738169,1169,106.325211,16347.665646,10.994570


In [28]:
# One row per flow, with all statistics computed in a single aggregation
flow_stats = grouped.agg(
    Bytes=("Length", "sum"),
    Packets=("Length", "size"),
    Start=("Timestamp", "min"),
    End=("Timestamp", "max"),
).reset_index()
flow_stats["Duration"] = flow_stats["End"] - flow_stats["Start"]

# Zero-duration flows (e.g. single packet) give NaN rates
dur = flow_stats["Duration"].replace(0, np.nan)
flow_stats["BytesPerSec"] = flow_stats["Bytes"] / dur
flow_stats["PacketsPerSec"] = flow_stats["Packets"] / dur

flow_stats.sort_values("Bytes", ascending=False).reset_index(drop=True)

,Source,SrcPort,Destination,DstPort,Bytes,Packets,Start,End,Duration,BytesPerSec,PacketsPerSec
0,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58506,52719449,34899,1.518337e+09,1.518337e+09,432.092108,122009.747511,80.767502
1,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58512,31358435,20855,1.518337e+09,1.518337e+09,277.847615,112861.990914,75.059129
2,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58498,25013186,16618,1.518337e+09,1.518337e+09,210.397020,118885.647658,78.984008
3,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58507,10650725,7106,1.518337e+09,1.518337e+09,49.454091,215365.903389,143.688820
4,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,443,192.168.43.72,58497,2320672,1583,1.518337e+09,1.518337e+09,114.381312,20288.908753,13.839673
...,...,...,...,...,...,...,...,...,...,...,...
293,192.168.43.72,56870,ns-vip-pro.paris.inria.fr,53,74,1,1.518337e+09,1.518337e+09,0.000000,NaN,NaN
294,192.168.43.72,56436,ns-vip-pro.paris.inria.fr,53,73,1,1.518337e+09,1.518337e+09,0.000000,NaN,NaN
295,192.168.43.72,11404,ns-vip-pro.paris.inria.fr,53,73,1,1.518337e+09,1.518337e+09,0.000000,NaN,NaN
296,192.168.43.72,41333,ns-vip-pro.paris.inria.fr,53,73,1,1.518337e+09,1.518337e+09,0.000000,NaN,NaN


## Note

Some of the libraries that we will use in this class, including the `netml` library from the University of Chicago, will compute these and other statistics automatically.

## Thought Questions

1. What are the largest flows in terms of: Number of bytes? Number of packets?

2. What do you notice about the flow sizes and the directions of flows?

3. What kinds of features are *not* available in packet summary statistics like those above which might be available in a raw packet trace? How might those features be useful for different packet classification problems?